In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from preprocessing import dt_profiles_rating_df, preprocess_text

C:\Users\jeffr\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ud-anthony.vpnstores.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\jeffr\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ud-anthony.vpnstores.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [2]:
### USER BASED FILLTERING ###
text_colums_user = ['gender','skin_type_face', 'hair_issue', 
                'skin_type_body', 'allergy_history', 'preferred_products', 
                'avoided_products', 'specific_needs']
                
for column in text_colums_user:
    dt_profiles_rating_df[column] = dt_profiles_rating_df[column].apply(preprocess_text)

In [3]:
# convert skin type, hair issue, skin type body to numeric value (int)
def convert_skin_type_face(skin_type): 
    skin_type_dict = {'normal': 0, 'kering': 1, 'minyak': 2, 'sensitif': 3, 'kombinasi': 4}
    return skin_type_dict.get(skin_type, 0)

def convert_hair_issue(hair_issue): 
    hair_issue_dict = {'normal': 1, 'ketombe': 1, 'kering': 2, 'minyak': 3, 'rontok': 4, 'cabang': 5}
    return hair_issue_dict.get(hair_issue, 0)

def convert_skin_type_body(skin_type): 
    skin_type_dict = {'normal': 0, 'kering': 1, 'minyak': 2, 'kombinasi': 3} 
    return skin_type_dict.get(skin_type, 0)

dt_profiles_rating_df["skin_type_face"] = dt_profiles_rating_df["skin_type_face"].apply(convert_skin_type_face) 
dt_profiles_rating_df["hair_issue"] = dt_profiles_rating_df["hair_issue"].apply(convert_hair_issue) 
dt_profiles_rating_df["skin_type_body"] = dt_profiles_rating_df["skin_type_body"].apply(convert_skin_type_body)

In [4]:
# Precompute user vectors
user_vectors = dt_profiles_rating_df.groupby('user_id')[['skin_type_face', 'hair_issue', 'skin_type_body']].mean().round(2)
user_vectors.reset_index(inplace=True)
user_vectors = user_vectors[user_vectors['user_id'].isin(dt_profiles_rating_df['user_id'].unique())]
user_ids = user_vectors['user_id']
user_vectors = user_vectors.drop('user_id', axis=1)
user_similarities = cosine_similarity(user_vectors)
user_similarities = pd.DataFrame(user_similarities, index=user_ids, columns=user_ids).round(2)

# Get unique items
items = dt_profiles_rating_df['product_id'].unique()

In [5]:
# Function to get recommendations
def get_user_based_recommendations(user_id):
    predictions = {}
    similarity_sum = user_similarities.loc[user_id].sum()
    
    if similarity_sum > 0:
        for item in items:
            other_user_ratings = dt_profiles_rating_df[dt_profiles_rating_df['product_id'] == item]
            rating_sum = 0
            weight_sum = 0
            for other_user_id in other_user_ratings['user_id']:
                if other_user_id != user_id:
                    rating = other_user_ratings[other_user_ratings['user_id'] == other_user_id]['rating'].values[0]
                    similarity = user_similarities.loc[user_id, other_user_id]
                    rating_sum += rating * similarity
                    weight_sum += similarity
            if weight_sum > 0:
                predictions[item] = rating_sum / weight_sum
            
    recommendations = sorted(predictions, key=predictions.get, reverse=True)[:10]
    return recommendations

# Display recommendations for each user
unique_user_ids = dt_profiles_rating_df['user_id'].unique()

In [6]:
for user_id in unique_user_ids:
    recommendations = get_user_based_recommendations(user_id)
    user_name = dt_profiles_rating_df[dt_profiles_rating_df['user_id'] == user_id]['user_name'].values[0]  # Assuming 'user_name' column exists
    print(f"Top 10 recommended products for user_id = {user_id} ({user_name})")
    print(f"Product IDs: {recommendations}")
    print()

Top 10 recommended products for user_id = 1 (Operator)
Product IDs: []

Top 10 recommended products for user_id = 2 (tester1)
Product IDs: [66, 195, 209, 153, 115, 183, 172, 61, 78, 39]

Top 10 recommended products for user_id = 3 (Elisa Regina Simanjuntak)
Product IDs: [66, 195, 227, 91, 277, 275, 209, 247, 119, 183]

Top 10 recommended products for user_id = 4 (Hamada)
Product IDs: [171, 12, 265, 267, 229, 291, 78, 312, 138, 95]

Top 10 recommended products for user_id = 5 (Elisa Regina Simanjuntak)
Product IDs: []

Top 10 recommended products for user_id = 6 (haruto)
Product IDs: []

Top 10 recommended products for user_id = 7 (Dini Sipahutar)
Product IDs: [66, 195, 209, 153, 115, 183, 172, 276, 61, 78]

Top 10 recommended products for user_id = 8 (Gladys)
Product IDs: [183, 78, 171, 260, 50, 33, 130, 232, 52, 86]

Top 10 recommended products for user_id = 9 (Suandika)
Product IDs: [302, 13, 33, 137, 195, 49, 52, 86, 209, 80]

Top 10 recommended products for user_id = 10 (Suandika N